In [24]:
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

MODEL_SAVE_PATH = '/content/drive/MyDrive/DR_Project/models'
DRIVE_DATA = '/content/drive/MyDrive/DR_Project/data/clinical'

# Reload the saved clinical model
clinical_model = joblib.load(f'{MODEL_SAVE_PATH}/clinical_model_xgboost.pkl')
clinical_threshold = joblib.load(f'{MODEL_SAVE_PATH}/clinical_threshold.pkl')

print("Clinical model loaded.")
print("Saved threshold:", clinical_threshold)

Clinical model loaded.
Saved threshold: 0.56


In [25]:
# Reload the raw clinical datasets
df_130 = pd.read_csv(f'{DRIVE_DATA}/diabetes_130_hospitals.csv')
df_diabd = pd.read_csv(f'{DRIVE_DATA}/diabd_bangladesh.csv')
df_pima = pd.read_csv(f'{DRIVE_DATA}/pima_indians.csv')

# Rebuild PIMA with expanded features (matching Step 5 from before)
pima_full = df_pima[['Age', 'Glucose', 'BMI', 'BloodPressure', 'Pregnancies',
                       'SkinThickness', 'Insulin', 'DiabetesPedigreeFunction', 'Outcome']].copy()
pima_full.columns = ['age', 'glucose', 'bmi', 'diastolic_bp', 'pregnancies',
                       'skin_thickness', 'insulin', 'pedigree_function', 'diabetic']
pima_full['diabetic'] = pima_full['diabetic'].map({1: 'Yes', 0: 'No'})
pima_full['gender'] = 'Female'
pima_full['systolic_bp'] = np.nan
pima_full['pulse_rate'] = np.nan
pima_full['family_diabetes'] = np.nan
pima_full['hypertensive'] = np.nan
pima_full['cardiovascular_disease'] = np.nan
pima_full['source'] = 'PIMA'

diabd_full = df_diabd[['age', 'glucose', 'bmi', 'diastolic_bp', 'systolic_bp',
                         'pulse_rate', 'gender', 'family_diabetes', 'hypertensive',
                         'cardiovascular_disease', 'diabetic']].copy()
diabd_full['pregnancies'] = np.nan
diabd_full['skin_thickness'] = np.nan
diabd_full['insulin'] = np.nan
diabd_full['pedigree_function'] = np.nan
diabd_full['source'] = 'DiaBD'

df_clinical_v2 = pd.concat([pima_full, diabd_full], ignore_index=True)

# Missing flags + imputation (same as before)
features_with_gaps = ['pregnancies', 'skin_thickness', 'insulin', 'pedigree_function',
                        'systolic_bp', 'pulse_rate', 'family_diabetes', 'hypertensive',
                        'cardiovascular_disease']
for col in features_with_gaps:
    df_clinical_v2[f'{col}_missing'] = df_clinical_v2[col].isnull().astype(int)
for col in features_with_gaps:
    df_clinical_v2[col] = df_clinical_v2[col].fillna(df_clinical_v2[col].median())

for col in ['glucose', 'bmi', 'diastolic_bp', 'skin_thickness', 'insulin']:
    mask = (df_clinical_v2[col] == 0) & (df_clinical_v2['source'] == 'PIMA')
    df_clinical_v2.loc[mask, col] = df_clinical_v2[df_clinical_v2['source']=='PIMA'][col].median()

df_clinical_v2['gender_encoded'] = LabelEncoder().fit_transform(df_clinical_v2['gender'])
df_clinical_v2['target'] = df_clinical_v2['diabetic'].map({'Yes': 1, 'No': 0})

feature_cols_v2 = ['age', 'glucose', 'bmi', 'diastolic_bp', 'gender_encoded',
                     'pregnancies', 'skin_thickness', 'insulin', 'pedigree_function',
                     'systolic_bp', 'pulse_rate', 'family_diabetes', 'hypertensive',
                     'cardiovascular_disease'] + [f'{c}_missing' for c in features_with_gaps]

X2 = df_clinical_v2[feature_cols_v2]
y2 = df_clinical_v2['target']

# Same split, same random_state — recreates the identical test set
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

# Get predictions from the reloaded model
clinical_test_proba = clinical_model.predict_proba(X2_test)[:, 1]

print("Clinical test set rebuilt:", X2_test.shape)
print("Clinical test predictions ready:", clinical_test_proba.shape)

/tmp/ipykernel_8043/61996007.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_130 = pd.read_csv(f'{DRIVE_DATA}/diabetes_130_hospitals.csv')


Clinical test set rebuilt: (1212, 23)
Clinical test predictions ready: (1212,)


In [27]:
np.random.seed(42)

# We already have:
# clinical_test_proba — shape (1212,) — clinical model's risk probabilities
# y_pred_proba_tta — shape (550, 5) — image model's per-class probabilities (with TTA)

n_pairs = min(len(clinical_test_proba), len(y_pred_proba_tta))
print(f"Creating {n_pairs} paired samples for fusion demonstration")

# Rank-based pairing: match by relative risk/severity position
# (not random — preserves realistic clinical logic: high clinical risk
# patients paired with more severe retinal images)
clinical_risk_rank = np.argsort(clinical_test_proba)
image_severity_scores = np.argmax(y_pred_proba_tta, axis=1)  # 0-4 severity per image
image_severity_rank = np.argsort(image_severity_scores)

clinical_indices = clinical_risk_rank[:n_pairs]
image_indices = image_severity_rank[:n_pairs]

paired_clinical_proba = clinical_test_proba[clinical_indices]
paired_image_proba = y_pred_proba_tta[image_indices]
paired_image_pred = np.argmax(paired_image_proba, axis=1)
paired_true_dr_stage = y_test_img[image_indices]  # true DR stage for evaluation later

print("Paired clinical risk range:", paired_clinical_proba.min(), "-", paired_clinical_proba.max())
print("\nPaired image severity distribution:")
print(pd.Series(paired_image_pred).value_counts().sort_index())

Creating 550 paired samples for fusion demonstration
Paired clinical risk range: 0.0111743 - 0.11747492

Paired image severity distribution:
0    275
1     40
2    175
3     27
4     33
Name: count, dtype: int64


In [30]:
import tensorflow.keras.backend as K

# Redefine the exact same focal loss function used during training
def categorical_focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1.0 - K.epsilon())
        cross_entropy = -y_true * K.log(y_pred)
        weight = alpha * K.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return K.sum(loss, axis=-1)
    return focal_loss

# Load the model, telling Keras what "focal_loss" refers to
image_model = load_model(
    f'{MODEL_SAVE_PATH}/mobilenetv2_final_v4_tta.keras',
    custom_objects={'focal_loss': categorical_focal_loss(gamma=2.0, alpha=0.25)}
)

print("Image model loaded successfully with custom loss function.")

y_pred_proba_tta = image_model.predict(X_test_img_fixed, verbose=0)
print("Image predictions ready:", y_pred_proba_tta.shape)

Image model loaded successfully with custom loss function.
Image predictions ready: (550, 5)


In [32]:
# Convert image model's 5-class output into a single "DR risk" severity score (0-1 scale)
# We do this by taking the weighted average across severity classes 0-4, normalized to 0-1
severity_weights = np.array([0, 1, 2, 3, 4])  # Stage 0 to Stage 4
image_severity_score = (y_pred_proba_tta[image_indices] @ severity_weights) / 4.0  # normalize to 0-1

# Late fusion: weighted combination (as specified in proposal — clinical 0.6, image 0.4)
CLINICAL_WEIGHT = 0.6
IMAGE_WEIGHT = 0.4

late_fusion_score = (CLINICAL_WEIGHT * paired_clinical_proba) + (IMAGE_WEIGHT * image_severity_score)

print("Late fusion score range:", late_fusion_score.min(), "-", late_fusion_score.max())
print("\nSample fusion scores (first 10):")
for i in range(10):
    print(f"Clinical risk: {paired_clinical_proba[i]:.3f} | Image severity: {image_severity_score[i]:.3f} | Fused: {late_fusion_score[i]:.3f}")

Late fusion score range: 0.008893619520222273 - 0.4274541156319174

Sample fusion scores (first 10):
Clinical risk: 0.011 | Image severity: 0.005 | Fused: 0.009
Clinical risk: 0.014 | Image severity: 0.002 | Fused: 0.009
Clinical risk: 0.014 | Image severity: 0.005 | Fused: 0.011
Clinical risk: 0.016 | Image severity: 0.005 | Fused: 0.011
Clinical risk: 0.016 | Image severity: 0.000 | Fused: 0.009
Clinical risk: 0.016 | Image severity: 0.001 | Fused: 0.010
Clinical risk: 0.016 | Image severity: 0.007 | Fused: 0.012
Clinical risk: 0.016 | Image severity: 0.001 | Fused: 0.010
Clinical risk: 0.016 | Image severity: 0.007 | Fused: 0.012
Clinical risk: 0.016 | Image severity: 0.044 | Fused: 0.027


In [33]:
from scipy.stats import pearsonr, spearmanr

# Correlation between fused score and true DR severity
pearson_corr, _ = pearsonr(late_fusion_score, paired_true_dr_stage)
spearman_corr, _ = spearmanr(late_fusion_score, paired_true_dr_stage)

print(f"Late Fusion — Pearson correlation with true DR stage: {pearson_corr:.4f}")
print(f"Late Fusion — Spearman correlation with true DR stage: {spearman_corr:.4f}")

# Also compare: how well does image-only score correlate (baseline for comparison)
pearson_image_only, _ = pearsonr(image_severity_score, paired_true_dr_stage)
print(f"\nImage-only severity score — Pearson correlation with true DR stage: {pearson_image_only:.4f}")

print(f"\nFusion improvement over image-only: {pearson_corr - pearson_image_only:+.4f}")

Late Fusion — Pearson correlation with true DR stage: 0.8755
Late Fusion — Spearman correlation with true DR stage: 0.8604

Image-only severity score — Pearson correlation with true DR stage: 0.8754

Fusion improvement over image-only: +0.0001


In [34]:
np.random.seed(42)

# Random pairing instead of rank-matched — removes the artificial correlation
random_clinical_idx = np.random.choice(len(clinical_test_proba), n_pairs, replace=False)
random_image_idx = np.random.choice(len(y_pred_proba_tta), n_pairs, replace=False)

random_paired_clinical = clinical_test_proba[random_clinical_idx]
random_paired_image_proba = y_pred_proba_tta[random_image_idx]
random_paired_true_stage = y_test_img[random_image_idx]

random_image_severity = (random_paired_image_proba @ severity_weights) / 4.0

random_late_fusion = (CLINICAL_WEIGHT * random_paired_clinical) + (IMAGE_WEIGHT * random_image_severity)

pearson_random_fusion, _ = pearsonr(random_late_fusion, random_paired_true_stage)
pearson_random_image_only, _ = pearsonr(random_image_severity, random_paired_true_stage)

print(f"Random pairing — Fusion correlation: {pearson_random_fusion:.4f}")
print(f"Random pairing — Image-only correlation: {pearson_random_image_only:.4f}")
print(f"Difference: {pearson_random_fusion - pearson_random_image_only:+.4f}")

Random pairing — Fusion correlation: 0.4683
Random pairing — Image-only correlation: 0.8754
Difference: -0.4071


In [38]:
image_feature_extractor = KerasModel(
    inputs=image_model.input,
    outputs=image_model.get_layer('global_average_pooling2d_1').output
)

image_embeddings = image_feature_extractor.predict(X_test_img_fixed, verbose=0)
print("Image embeddings shape:", image_embeddings.shape)

Image embeddings shape: (550, 1280)


In [39]:
# Step 6 — Clinical embeddings
clinical_embeddings = X2_test.values
print("Clinical embeddings shape:", clinical_embeddings.shape)

# Step 7 — Paired fusion dataset
paired_image_embeddings = image_embeddings[image_indices]
paired_clinical_embeddings = clinical_embeddings[clinical_indices]
fused_features = np.concatenate([paired_clinical_embeddings, paired_image_embeddings], axis=1)
print("Fused feature vector shape:", fused_features.shape)

# Step 8 — Build intermediate fusion model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model as KerasModel2
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split as tts

y_fusion_target = to_categorical(paired_true_dr_stage, num_classes=5)

X_fusion_train, X_fusion_test, y_fusion_train, y_fusion_test = tts(
    fused_features, y_fusion_target, test_size=0.3, random_state=42
)

fusion_input = Input(shape=(fused_features.shape[1],))
x = Dense(128, activation='relu')(fusion_input)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
fusion_output = Dense(5, activation='softmax')(x)

intermediate_fusion_model = KerasModel2(inputs=fusion_input, outputs=fusion_output)
intermediate_fusion_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

intermediate_fusion_model.summary()

Clinical embeddings shape: (1212, 23)
Fused feature vector shape: (550, 1303)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1303)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       166,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,493 (685.52 KB)

 Trainable params: 175,493 (685.52 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
from tensorflow.keras.callbacks import EarlyStopping

fusion_callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
]

history_fusion = intermediate_fusion_model.fit(
    X_fusion_train, y_fusion_train,
    validation_data=(X_fusion_test, y_fusion_test),
    batch_size=16,
    epochs=100,
    callbacks=fusion_callbacks,
    verbose=1
)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 162ms/step - accuracy: 0.3766 - loss: 2.9226 - val_accuracy: 0.6424 - val_loss: 1.0365
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5247 - loss: 1.3549 - val_accuracy: 0.7212 - val_loss: 0.8587
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6468 - loss: 1.0146 - val_accuracy: 0.7030 - val_loss: 0.7979
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6649 - loss: 0.9091 - val_accuracy: 0.7152 - val_loss: 0.7420
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6909 - loss: 0.8787 - val_accuracy: 0.7212 - val_loss: 0.7095
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7247 - loss: 0.7530 - val_accuracy: 0.7273 - val_loss: 0.6917
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7325 - loss: 0.7712 - val_accuracy: 0.7212 - val_loss: 0.6847
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7584 - loss: 0.7065 - val_accuracy: 0.6909 -

In [41]:
y_pred_proba_intfusion = intermediate_fusion_model.predict(X_fusion_test, verbose=0)
y_pred_intfusion = np.argmax(y_pred_proba_intfusion, axis=1)
y_true_fusion_test = np.argmax(y_fusion_test, axis=1)

test_accuracy_intfusion = np.mean(y_pred_intfusion == y_true_fusion_test)
qwk_intfusion = cohen_kappa_score(y_true_fusion_test, y_pred_intfusion, weights='quadratic')

print("=== Intermediate Fusion Results ===")
print(f"Test Accuracy: {test_accuracy_intfusion:.4f}")
print(f"Quadratic Weighted Kappa: {qwk_intfusion:.4f}")
print("\n", classification_report(y_true_fusion_test, y_pred_intfusion, target_names=['Stage 0','Stage 1','Stage 2','Stage 3','Stage 4']))

=== Intermediate Fusion Results ===
Test Accuracy: 0.7273
Quadratic Weighted Kappa: 0.7524

               precision    recall  f1-score   support

     Stage 0       0.97      0.95      0.96        81
     Stage 1       0.50      0.07      0.12        14
     Stage 2       0.51      0.93      0.66        45
     Stage 3       0.00      0.00      0.00         9
     Stage 4       0.00      0.00      0.00        16

    accuracy                           0.73       165
   macro avg       0.40      0.39      0.35       165
weighted avg       0.66      0.73      0.66       165



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [42]:
intermediate_fusion_model.save(f'{MODEL_SAVE_PATH}/intermediate_fusion_model.keras')

import json
fusion_results = {
    'late_fusion': {
        'rank_paired_correlation': 0.8755,
        'random_paired_correlation': 0.4683
    },
    'intermediate_fusion': {
        'test_accuracy': float(test_accuracy_intfusion),
        'qwk': float(qwk_intfusion),
        'training_samples': 385,
        'note': 'Underperformed image-only due to limited paired training data; zero recall on Stage 3/4 due to class scarcity'
    },
    'image_only_baseline': {
        'test_accuracy': 0.780,
        'auc_roc': 0.920,
        'qwk': 0.863
    }
}
with open(f'{MODEL_SAVE_PATH}/fusion_ablation_results.json', 'w') as f:
    json.dump(fusion_results, f, indent=2)

print("Fusion ablation study complete and saved.")

Fusion ablation study complete and saved.


In [43]:
import os
print("Files in your models folder:")
for f in os.listdir(MODEL_SAVE_PATH):
    print(" -", f)

Files in your models folder:
 - clinical_model_xgboost.pkl
 - clinical_threshold.pkl
 - shap_summary.png
 - shap_waterfall_patient1.png
 - mobilenetv2_phase1_best.keras
 - mobilenetv2_phase2_best.keras
 - mobilenetv2_phase3_balanced.keras
 - mobilenetv2_final.keras
 - model2_final_results.json
 - gradcam_sample.png
 - gradcam_severe_examples.png
 - mobilenetv2_focal_loss.keras
 - mobilenetv2_final_v4_tta.keras
 - intermediate_fusion_model.keras
 - fusion_ablation_results.json


In [45]:
demo_idx = 5  # same index as before

demo_clinical_features = X2_test.iloc[clinical_indices[demo_idx]]
demo_clinical_risk = paired_clinical_proba[demo_idx]
demo_image_idx_in_test = image_indices[demo_idx]
demo_true_stage = y_test_img[demo_image_idx_in_test]
demo_predicted_stage = np.argmax(y_pred_proba_tta[demo_image_idx_in_test])
demo_image_confidence = y_pred_proba_tta[demo_image_idx_in_test][demo_predicted_stage]

print("=== Demo Patient Summary ===")
print("Clinical risk score:", demo_clinical_risk)
print("True DR stage:", demo_true_stage)
print("Predicted DR stage:", demo_predicted_stage)
print("Prediction confidence:", demo_image_confidence)
print("\nClinical features:")
print(demo_clinical_features)

=== Demo Patient Summary ===
Clinical risk score: 0.015670525
True DR stage: 0
Predicted DR stage: 0
Prediction confidence: 0.9965359

Clinical features:
age                                35.0000
glucose                             7.4500
bmi                                16.4800
diastolic_bp                       71.0000
gender_encoded                      0.0000
pregnancies                         3.0000
skin_thickness                     23.0000
insulin                            30.5000
pedigree_function                   0.3725
systolic_bp                       114.0000
pulse_rate                         72.0000
family_diabetes                     0.0000
hypertensive                        0.0000
cardiovascular_disease              0.0000
pregnancies_missing                 1.0000
skin_thickness_missing              1.0000
insulin_missing                     1.0000
pedigree_function_missing           1.0000
systolic_bp_missing                 0.0000
pulse_rate_missing           

In [46]:
import os
os.makedirs('/content/report_output', exist_ok=True)

patient_data = {
    'age': int(demo_clinical_features['age']),
    'glucose': round(float(demo_clinical_features['glucose']), 1),
    'bmi': round(float(demo_clinical_features['bmi']), 1),
    'diastolic_bp': int(demo_clinical_features['diastolic_bp']),
    'clinical_risk_score': round(float(demo_clinical_risk) * 100, 1),
    'true_dr_stage': int(demo_true_stage),
    'predicted_dr_stage': int(demo_predicted_stage),
    'image_confidence': round(float(demo_image_confidence) * 100, 1),
}

stage_labels = {0: 'No DR', 1: 'Mild NPDR', 2: 'Moderate NPDR', 3: 'Severe NPDR', 4: 'Proliferative DR'}
patient_data['predicted_stage_label'] = stage_labels[patient_data['predicted_dr_stage']]

print(patient_data)

{'age': 35, 'glucose': 7.5, 'bmi': 16.5, 'diastolic_bp': 71, 'clinical_risk_score': 1.6, 'true_dr_stage': 0, 'predicted_dr_stage': 0, 'image_confidence': 99.7, 'predicted_stage_label': 'No DR'}


In [47]:
img_array = np.expand_dims(X_test_img_fixed[demo_image_idx_in_test], axis=0)
original_img = np.uint8((X_test_img[demo_image_idx_in_test]) * 255)

heatmap, _ = make_gradcam_heatmap(img_array, image_model, 'Conv_1_bn')
overlayed = overlay_gradcam(original_img, heatmap)

plt.figure(figsize=(4,4))
plt.imshow(overlayed)
plt.axis('off')
plt.savefig('/content/report_output/patient_gradcam.png', dpi=150, bbox_inches='tight', pad_inches=0)
plt.close()

print("Grad-CAM image saved for report.")

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Grad-CAM image saved for report.


In [49]:
import shap
explainer = shap.TreeExplainer(clinical_model)
print("SHAP explainer rebuilt.")

SHAP explainer rebuilt.


In [50]:
clinical_idx_in_test = clinical_indices[demo_idx]

shap_values_patient = explainer.shap_values(X2_test.iloc[[clinical_idx_in_test]])

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values_patient[0],
        base_values=explainer.expected_value,
        data=X2_test.iloc[clinical_idx_in_test],
        feature_names=feature_cols_v2
    ),
    show=False
)
plt.tight_layout()
plt.savefig('/content/report_output/patient_shap.png', dpi=150, bbox_inches='tight')
plt.close()

print("SHAP chart saved for report.")

SHAP chart saved for report.
